## KG BUILDER

In [1]:
import os
import pandas as pd
import requests

from neo4j import GraphDatabase
from graphdatascience import GraphDataScience

/home/ndionysus/anaconda3/envs/ngsi/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Neo4j connection details
uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
username = os.environ.get("NEO4J_USERNAME", "neo4j")
password = os.environ["NEO4J_PASSWORD"]
driver = GraphDatabase.driver(uri, auth=(username, password))


# SATO 
api_endpoints = ["https://project-sato1.lasige.di.fc.ul.pt/api", "https://project-sato2.lasige.di.fc.ul.pt/api"]
headers = {"x-api-key": os.environ["SATO_API_KEY"]}

#### SATO functions

In [5]:
def sato_query(query):
    for i in range(2):
        payload = requests.get(api_endpoints[i] + query, headers=headers)
        if(payload.status_code == 200): 
            return payload
    else:
        return None 


def list_entities(entity_type, idPattern=None, q=None, options=None):
    endpoint = type_to_entrypoint(entity_type=entity_type)
    query = f"/{endpoint}?"

    if (idPattern is not None):
        query += f"idPattern=.*{idPattern}.*"
    if (q is not None):
        query += f"&q={q}"
    if (options is not None):
        query += f"&options={options}"

    offset = 0
    all_entities = []
    while (True):
        entity = sato_query(query + f"&offset={offset}")
        if (entity.status_code == 200):
            entities = entity.json()
            if (len(entities) == 0):
                return all_entities
            else:
                all_entities += entities
            offset += 100
    else:
        raise Exception(f"FAILED to get entity info: {query}")

def type_to_entrypoint(entity_type):
    entity_to_endpoint = {
        "Device": "devices",
        "Building": "buildings",
        "Zone": "zones",
        "Space": "spaces",
        "Site": "sites",
        "External": "external",
        "Task": "tasks"
    }
    return entity_to_endpoint[entity_type]


def create_node(tx, query, params):
    tx.run(query, **params)

### Create all entities

In [7]:
# entity_types = ["Building", "Space", "Zone", "Device", "External", "Task"]
entity_types = ["Task"]

In [4]:
with driver.session() as session:
    for entity_type in entity_types:
        print("Fetching all entities of type:", entity_type)
        entities = list_entities(entity_type=entity_type, options="full")

        print("Populating graph...")
        for entity in entities:
            
            # Init with ID and Type as they are special properties
            entity_id = entity["id"]
            if("Berkeley" in entity_id): continue
            
            entity_type = entity["type"]
            params = {"id": entity_id, "type": entity_type}

            # Build the query depending on existing data
            query = [f"CREATE (n: {entity_type}", "{id: $id, type: $type"]

            for prop, value in entity.items():
                # Skip ID and Type since we already have them above
                if(prop in ["id", "type"]): continue

                # If the prop is a list, then its a relationship and we can skip it
                if (type(value) == list):
                    continue
                if (type(value) != dict):
                    raise Exception("Yo wtf? There is prop that is not a dict besides ID and Type?")

                # Now that we are sure its a dict, lets make sure it is not a relationship
                if (value["type"] == "Relationship" or prop == "requires"):
                    continue

                # Finally Add the property
                query += [f",{prop}: ${prop}"]
                params[prop] = str(value["value"])

            # Close the query
            query += ["})"]
            

            final_query = "".join(query)
            session.execute_write(create_node, final_query, params)

Fetching all entities of type: Building


NameError: name 'list_entities' is not defined

### Link entities using Relationships

In [14]:
with driver.session() as session:
    for entity_type in entity_types:
        print("Fetching all entities of type:", entity_type)
        entities = list_entities(entity_type=entity_type, options="full")

        print("Populating graph...")
        for entity in entities:

            # Init with ID and Type as they are special properties
            entity_id = entity["id"]
            if ("Berkeley" in entity_id):
                continue

            entity_type = entity["type"]

            for prop, value in entity.items():
                # Skip ID and Type
                if (prop in ["id", "type"]):
                    continue

                # If the prop is a list, then its a relationship
                if (type(value) == list):
                    for target_entity in value:
                        target_id = target_entity["object"]
                        target_type = target_id.split(":")[2]
                        
                        print(entity_id, "--", prop, "-->", target_id)

                        query = f"MATCH (src: {entity_type} " + "{id: $src_id}),"
                        query += f"(tgt: {target_type}" + "{id: $tgt_id}) "
                        query += f"CREATE (src)-[:{prop}]->(tgt)"

                        session.execute_write(create_node, query, {"src_id": entity_id, "tgt_id": target_id})
                        
                elif(value["type"] == "Relationship"):
                    target_id = value["object"]
                    target_type = target_id.split(":")[2]

                    print(entity_id, "--", prop, "-->", target_id)
                    
                    query = f"MATCH (src: {entity_type} " + "{id: $src_id}),"
                    query += f"(tgt: {target_type}" + "{id: $tgt_id}) "
                    query += f"CREATE (src)-[:{prop}]->(tgt)"

                    session.execute_write(create_node, query, {"src_id": entity_id, "tgt_id": target_id})

                # TODO: Add a new elif for when prop == "requires"
                elif(prop == "requires"):
                    for req_k, req_v in value["value"].items():
                        if("id" not in req_v): continue
                        full_id = "urn:ngsi-ld:" + req_v["type"] + ":" + req_v["id"]
                        rel = ("reqAs_" + req_k).replace("-", "_")
                        print(full_id, rel)

                        query = f"MATCH (src: {entity_type} " + "{id: $src_id}),"
                        query += f"(tgt: {req_v['type']}" + "{id: $tgt_id}) "
                        query += f"CREATE (src)-[:{rel}]->(tgt)"

                        session.execute_write(create_node, query, {
                                            "src_id": entity_id, "tgt_id": full_id})

Fetching all entities of type: Task
Populating graph...
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Exhaust-Fan-Flow reqAs_ExtFanFlow
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Exhaust-Fan-Power reqAs_ExtFanPower
urn:ngsi-ld:Device:Aalborg-Office:Weather-Station:Sensor_Temperature reqAs_ExtTemp
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Supply-Fan-Flow reqAs_SupFanFlow
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Supply-Fan-Power reqAs_SupFanPower
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Temperature-HE reqAs_AfterHETemp
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Exhaust-Fan-Flow reqAs_ExtFanFlow
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Temperature-Extraction reqAs_ExtractionAirTemp
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Temperature-Intake reqAs_IntakeAirTemp
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Sensor_Supply-Fan-Flow reqAs_SupFanFlow
urn:ngsi-ld:Device:Aalborg-Office:HVAC:Actuator_Switch-Supply-Fan-OnOff reqAs_SupFanOnOff
urn:ngsi-ld:Device:Aalborg-Office